# Quantization Troubleshooting with the Model Compression Toolkit (MCT) Using the XQuant Extension Tool

[Run this tutorial in Google Colab(リンクはまだ)](https://colab.research.google.com/github/SonySemiconductorSolutions/mct-model-optimization/blob/main/tutorials/notebooks/mct_features_notebooks/pytorch/example_pytorch_xquant.ipynb)

## Overview
This notebook provides valuable insights into improving the quality of the quantization process for PyTorch models using the XQuant extension tool.It calculates the error for each layer by comparing the floating-point model and the quantized model, along with the quantization log. The results are presented in a report format, identifying the causes of the detected errors and suggesting appropriate improvement measures for each cause.

## Summary
We will cover the following steps:

1. Load a pre-trained MobileNetV3 model and perform post-training quantization.
2. XQuant Extension Tool
3. outlier removal
4. Representative dataset size & diversit
5. biass correction
6. using more samples in mixed precision quantization
7. threshold selection error method
8. enabling hessian based mixed precision
9. GPTQ - Gradient-Based Post Training Quantization
    
## Setup
Install the relevant packages:

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0

In [ ]:
import importlib
if not importlib.util.find_spec('model_compression_toolkit'):
    !pip install model_compression_toolkit

In [ ]:
from functools import partial
from model_compression_toolkit.xquant import XQuantConfig
import torch

## Define a Random Data Generator
For demonstration purposes, we will use a random dataset generator to create both the representative dataset and the validation dataset. This will allow us to simulate data for quantization and validation without using an actual dataset.

In [ ]:
# Function to generate random data. If use_labels is True, it yields data with labels;
# otherwise, it yields only data.
def random_data_gen(shape=(3, 224, 224), use_labels=False, batch_size=2, num_iter=2):
    if use_labels:
        for _ in range(num_iter):
            yield [[torch.randn(batch_size, *shape)], torch.randn(batch_size)]
    else:
        for _ in range(num_iter):
            yield [torch.randn(batch_size, *shape)]

## Post-Training Quantization using MCT (no correction)

In [ ]:
# Load the pre-trained MobileNetV2 model and perform post-training quantization using
# the representative dataset generated by random_data_gen.
from torchvision.models.mobilenetv2 import MobileNetV2
import model_compression_toolkit as mct

float_model = MobileNetV2()
quantized_model, quantized_info = mct.ptq.pytorch_post_training_quantization(
    in_module=float_model, representative_data_gen=random_data_gen)

## Use the XQuant Extension Tool

In [ ]:
mct.set_log_folder('./log_XQuant_Extension_Tools')

# Define the validation dataset and Xquant configuration, including custom similarity metrics.
validation_dataset = partial(random_data_gen, use_labels=True)
xquant_config = XQuantConfig(report_dir='./log_xquant')

from model_compression_toolkit.xquant import xquant_report_troubleshoot_pytorch_experimental
result = xquant_report_troubleshoot_pytorch_experimental(
            float_model,
            quantized_model,
            random_data_gen,
            validation_dataset,
            xquant_config
        )

## Understanding the Quantization Error Graph

Six quantization error graphs will be generated in the directory specified by report_dir in XQuantConfig.
They show three metrics (MSE, cosine similarity, and SQNR) for two datasets (representative dataset and validation dataset).
The quantization error represents the difference in layer output between the floating-point model and the quantized model.

Comparing each quantization error with threshold_quantize_error to identify layers with significant behavior changes after quantization.

As an example, an output graph calculated using “mse” with a representative dataset is shown. The initial threshold value of 0.1 is set, and layers exceeding this threshold are indicated with a red circle. In addition, the corresponding layer names on the X axis are highlighted in red. With this graph, layers with accuracy degradation can be visually confirmed.



picture  ---   quant_loss_mse_repr.png



X-axis: Layer names (layers identified as degraded are highlighted in red)
Y-axis: Quantization error
Red dashed line: Threshold for accuracy degradation as set in XQuantConfig
Red circle: Layers judged to have degraded accuracy

## Quantization Troubleshooting for MCT

The Model Compression Toolkit (MCT) offers numerous functionalities to compress neural networks with minimal accuracy lost. However, in some cases, the compressed model may experience a significant decrease in accuracy. Fear not, as this lost accuracy can often be reclaimed by adjusting the quantization configuration or setup.
Outlined below are a series of steps aimed at recovering lost accuracy resulting from compression with MCT. Some steps may be applicable to your model, while others may not.
XQuant Extension Tool will automatically detect the problem and display the appropriate warning message on the console. Please refer to the respective troubleshooting manual and change the settings if necessary.

see [TroubleShooting Manual](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/index.html)


## outlier removal
Outlier removal can become essential when quantizing activations, particularly in scenarios where certain layers produce output activation tensors with skewed value distributions.
The quantization accuracy may degrade when there are outlier activations in the quantized layers of your model.
You can check if there are any outliers in your activation tensor by visualizing the histogram (shown below) generated in the directory specified by report_dir in XQuantConfig.
Manually limit the activation thresholds using the z_threshold attribute of QuantizationConfig in CoreConfig.
Set z_threshold to a value. Typical value range is between 5.0 and 20.0.


See [TroubleShooting Documentation>>Outlier Removal](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/outlier_removal.html#ug-outlier-removal)

In [ ]:
core_config = mct.core.CoreConfig(mct.core.QuantizationConfig(z_threshold=3.9))
quantized_model, quantized_info = mct.ptq.pytorch_post_training_quantization(
    in_module=float_model, representative_data_gen=random_data_gen)

## General Troubleshoots
If there is no significant improvement, comprehensively evaluate other areas for improvement.
The following items are general troubleshoots for quantization accuracy improvement.

## Representative dataset size & diversity
The representative dataset is used by the MCT to derive the threshold values of activation tensors in the model.
If the representative dataset size is too small, the thresholds will overfit and the accuracy on the validation dataset will degrade.

A similar overfitting may occur when the representative dataset isn’t diverse enough (e.g. images from a single class, in a classification model). In this case, the distribution of the target dataset and the representative dataset will not match, which might cause an accuracy degradation.

Increase the number of samples in the representative dataset.
Make sure that the samples are more diverse (e.g. include samples from all the classes, in a classification model).

## biass correction
MCT applies bias correction by default to overcome induced bias shift caused by weights quantization.
The applied correction is an estimation of the bias shift that is computed based on the collected statistical data generated with the representative dataset.
Therefore, the effect of the bias correction is sensitive to the distribution and size of the provided representative dataset.

The quantization accuracy may degrade when the representative dataset and bias correction are incompatible, causing a decrease in accuracy.

You can check if the bias correction causes a degradation in accuracy, by disabling the bias correction (setting weights_bias_correction to False of the QuantizationConfig in CoreConfig).

  1 If you can increase your representative dataset size and its distribution, it may restore accuracy.  
  2 If you don’t have an option to increase or diversify your representative dataset, disabling the bias   

see [TroubleShooting Documentation>>Bias Correction](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/bias_correction.html#ug-bias-correction)




## using more samples in mixed precision quantization


In Mixed Precision quantization, MCT will assign a different bit width to each weight in the model, depending on the weight’s layer sensitivity and a resource constraint defined by the user, such as target model size.
Check out the mixed precision tutorial for more information.

By default, MCT employs 32 samples from the provided representative dataset for the Mixed Precision search. Leveraging a larger dataset could enhance results, particularly when dealing with datasets exhibiting high variance.

The quantization accuracy may degrade when using Mixed Precision quantization with a small number of samples.

Solution¶
Set the num_of_images attribute to a larger value of the MixedPrecisionQuantizationConfig in CoreConfig.

see [TroubleShooting Documentation>>Using more samples in Mixed Precision quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/using_more_samples_in_mixed_precision_quantization.html#ug-using-more-samples-in-mixed-precision-quantization)


## threshold selection error method

The quantization threshold, which determines how data gets quantized, involves an optimization process driven by predefined objective metrics.
MCT defaults to employing the Mean-Squared Error (MSE) metric for threshold optimization,
however, it offers a range of alternative error metrics (e.g. using min/max values, KL-divergence, etc.) to accommodate different network requirements.
This flexibility becomes particularly crucial for activation quantization, where threshold selection spans the entire tensor and relies on statistical insights for optimization.
We advise you to consider other error metrics if your model is suffering from significant accuracy degradation, especially if it contains unorthodox activation layers.

Use a different error method for activations. You can set the following values:
  NOCLIPPING - Use min/max values
  MSE (default) - Use mean square error
  MAE - Use mean absolute error
  KL - Use KL-divergence
  Lp - Use Lp-norm
  HMSE - Use Hessian-based mean squared error

For example, set NOCLIPPING to the activation_error_method attribute of the QuantizationConfig in CoreConfig.

see [TroubleShooting Documentation>>Threshold selection error method](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/threhold_selection_error_method.html#ug-threshold-selection-error-method)

## enabling hessian based mixed precision
In Mixed Precision quantization, MCT will assign a different bit width to each weight in the model, depending on the weight’s layer sensitivity and a resource constraint defined by the user, such as target model size.

MCT offers a Hessian-based scoring mechanism to assess the importance of layers during the Mixed Precision search.
This feature can notably enhance Mixed Precision outcomes for certain network architectures.

Set the use_hessian_based_scores flag to True in the MixedPrecisionQuantizationConfig of the CoreConfig.

see [TroubleShooting Documentation>>Enabling Hessian-based Mixed Precision](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/enabling_hessian-based_mixed_precision.html#ug-enabling-hessian-based-mixed-precision)


## GPTQ - Gradient-Based Post Training Quantization
When PTQ (either with or without Mixed Precision) fails to deliver the required accuracy, GPTQ is potentially the remedy.
In GPTQ, MCT will finetune the model’s weights and quantization parameters for improved accuracy. The finetuning process will only use the label-less representative dataset.
Check out the GPTQ tutorial for more information and an implementation example.

MCT can configure GPTQ optimization options, such as the number of epochs for the optimization process.
For example, set the number of epochs to 50.

see [GPTQ - Gradient-Based Post Training Quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/gptq-gradient_based_post_training_quantization.html#ug-gptq-gradient-based-post-training-quantization)

## Copyrights
Copyright 2026 Sony Semiconductor Solutions, Inc. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
